In [11]:
%pip install faiss-cpu
%pip install faiss-gpu
import faiss
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


Note: you may need to restart the kernel to use updated packages.


c:\Users\patel\anaconda3\envs\RP_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings = np.load("data/processed/job_embeddings.npy")

In [3]:
embeddings.shape

(123849, 384)

In [4]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

In [5]:
index.add(embeddings)

In [6]:
print(index.ntotal)

123849


In [7]:
k = 5
query_vector = embeddings[100].reshape(1, -1)
Distances, Indices = index.search(query_vector, k)

In [8]:
print(Indices)

[[   100   5740  46043 113992   6317]]


In [9]:
jobs = pd.read_csv("data/processed/jobs_preprocessed.csv")

In [10]:
jobs.iloc[Indices[0]][['title']]

,title
100,Construction Project Manager
5740,Construction Project Manager
46043,Senior Project Manager- Education
113992,Construction Project Manager
6317,Construction Senior Project Manager - Data Cen...


In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')
query = 'Data Scientist with experience in Python and Machine Learning'
query_embedding = model.encode([query])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4048.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
Distances, Indices = index.search(query_embedding, k)
jobs.iloc[Indices[0]][['title']]

,title
70619,Machine Learning Engineer/Python Developer
93501,Data Scientist
4767,Developer
6478,Python Developer
70991,Lead Python Software Engineer


In [14]:
query = "Experience with java, c++, python"
query_embedding = model.encode([query])
Distances, Indices = index.search(query_embedding, k)
jobs.iloc[Indices[0]][['title']]

,title
30129,Python API Developer (Python Binding for C++Li...
83974,Python/C++ Software Engineer
3848,"C++ Developer with Python - Remote @ Sandiego, CA"
54880,"C++ Developer with Python - Remote @ Sandiego, CA"
84872,"C++ Developer with Python - Remote @ Sandiego, CA"


In [15]:
query = "Experience with sql, databases"
query_embedding = model.encode([query])
Distances, Indices = index.search(query_embedding, k)
jobs.iloc[Indices[0]][['title']]

,title
75515,Database Developer
75628,Database Developer
6363,SQL Developer
65959,Database Administrator
73323,SQL Database Administrator


In [16]:
faiss.write_index(index, "data/processed/faiss_index.bin")